# SongFormer: Music Structure Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SidSaxena/SongFormer/blob/cross-platform/notebooks/SongFormer-Colab.ipynb)

**SongFormer** is a music structure analysis framework that segments songs into labeled sections (intro, verse, chorus, bridge, etc.).

This notebook provides two options:
- **Option A:** Run the Gradio web app (interactive, upload audio and see results)
- **Option B:** Run batch inference (process multiple audio files at once)

**Requirements:** A GPU runtime is recommended. Go to `Runtime → Change runtime type → GPU`.

Paper: [arXiv:2510.02797](https://arxiv.org/abs/2510.02797) | GitHub: [ASLP-lab/SongFormer](https://github.com/ASLP-lab/SongFormer)

---
## 1. Check GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU Memory: {mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

---
## 2. Clone repository and install dependencies

In [ ]:
!git clone https://github.com/SidSaxena/SongFormer.git -b cross-platform
%cd SongFormer
!git submodule update --init --recursive

In [ ]:
!pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -r requirements.txt

---
## 3. Download model checkpoints

Checkpoints are **~1.33 GB** total:
- `pretrained_msd.pt` (MusicFM) — 1.23 GB
- `SongFormer.safetensors` — 99.6 MB
- `msd_stats.json` — 2.2 KB

Persisting on Drive saves bandwidth across sessions, but uses ~1.33 GB of Drive space.

In [ ]:
#@markdown ### Checkpoint Storage
#@markdown Persist checkpoints on Google Drive to avoid re-downloading each session?
STORE_ON_DRIVE = False  #@param {type:"boolean"}

import os
import shutil

REPO_CKPTS = '/content/SongFormer/src/SongFormer/ckpts'

if STORE_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPTS = '/content/drive/MyDrive/SongFormer/ckpts'
    os.makedirs(DRIVE_CKPTS, exist_ok=True)

    # Symlink Drive checkpoints into the repo
    if os.path.isdir(REPO_CKPTS) and not os.path.islink(REPO_CKPTS):
        for f in os.listdir(REPO_CKPTS):
            src = os.path.join(REPO_CKPTS, f)
            dst = os.path.join(DRIVE_CKPTS, f)
            if not os.path.exists(dst):
                if os.path.isdir(src):
                    shutil.copytree(src, dst)
                else:
                    shutil.copy2(src, dst)
        shutil.rmtree(REPO_CKPTS)
    elif not os.path.exists(REPO_CKPTS):
        pass

    if not os.path.islink(REPO_CKPTS):
        os.symlink(DRIVE_CKPTS, REPO_CKPTS)
    print(f"Checkpoints will be stored at: {DRIVE_CKPTS}")
else:
    print("Checkpoints will be downloaded to the session (not persisted).")

# Download any missing checkpoints
os.chdir('/content/SongFormer/src/SongFormer')
from utils.fetch_pretrained import download_all
download_all()
os.chdir('/content/SongFormer')

# Verify
print("\nDownloaded checkpoints:")
for f in ['MusicFM/msd_stats.json', 'MusicFM/pretrained_msd.pt', 'SongFormer.safetensors']:
    path = os.path.join(REPO_CKPTS, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

---
## 4. Audio Source

Choose where your audio files come from:
- **Google Drive folder** — enter a path below, files are copied automatically
- **Upload manually** — leave empty, upload in the next cell

Supports full paths (`/content/drive/MyDrive/my_songs`) or relative to Drive root (`my_songs`).

Audio formats: `.mp3`, `.wav`, `.flac`, `.ogg`, `.m4a`, `.aac`, `.wma`

In [ ]:
#@markdown ### Audio Source
#@markdown Enter a Google Drive folder path, or leave empty to upload manually.
DRIVE_AUDIO_FOLDER = ""  #@param {type:"string"}

import os
import shutil

AUDIO_DIR = '/content/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')

drive_audio_available = False

if DRIVE_AUDIO_FOLDER.strip():
    # Mount Drive if not already mounted
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')

    # Resolve path (support both full and relative paths)
    src = DRIVE_AUDIO_FOLDER.strip()
    if not src.startswith('/content/'):
        src = f'/content/drive/MyDrive/{src}'

    if os.path.isdir(src):
        count = 0
        for f in sorted(os.listdir(src)):
            if f.lower().endswith(AUDIO_EXTENSIONS):
                shutil.copy2(os.path.join(src, f), AUDIO_DIR)
                count += 1
        if count > 0:
            print(f"Copied {count} audio files from: {src}")
            drive_audio_available = True
        else:
            print(f"No audio files found in: {src}")
            print(f"Supported formats: {', '.join(AUDIO_EXTENSIONS)}")
    else:
        print(f"Directory not found: {src}")
        print("Please check the path and run this cell again.")
else:
    print("No Drive folder specified.")
    print("Upload audio files in the next cell, or use Gradio's upload UI.")

# Show current audio files
audio_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
]) if os.path.isdir(AUDIO_DIR) else []

if audio_files:
    print(f"\nAudio files in {AUDIO_DIR} ({len(audio_files)} total):")
    for f in audio_files:
        size_mb = os.path.getsize(os.path.join(AUDIO_DIR, f)) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")

---
## Option A: Gradio Web App

Run the interactive Gradio app. Colab will generate a public URL you can open in your browser.

**How to use:**
1. Run the cell below
2. Click the public URL that appears (e.g., `https://xxxxx.gradio.live`)
3. Upload an audio file (or use files from the audio directory if you specified a Drive folder)
4. Click "Analyze Music Structure"

If you specified a Drive folder in step 4, those files are available at `/content/audio/`.

In [ ]:
%cd /content/SongFormer
!python app.py

---
## Option B: Batch Inference

Process multiple audio files at once using the command-line inference script.

**How to use:**
1. Configure audio source in step 4 (Drive folder or upload below)
2. Create an `.scp` file listing the paths
3. Run inference
4. Download results

### B1. Upload audio files

Skip this cell if you already specified a Drive folder in step 4.

In [ ]:
#@markdown ### Upload Audio Files
#@markdown Skip if you already specified a Drive folder above.

import os

AUDIO_DIR = '/content/audio'
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')

# Check if audio files already exist (from Drive)
existing_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
]) if os.path.isdir(AUDIO_DIR) else []

if existing_files:
    print(f"Already have {len(existing_files)} audio files from Drive:")
    for f in existing_files:
        print(f"  {f}")
    print("\nUpload more files below, or skip this cell.")

# Upload widget
from google.colab import files
print("\nUpload audio files (mp3, wav, flac, etc.):")
uploaded = files.upload()
for filename, data in uploaded.items():
    dst = os.path.join(AUDIO_DIR, filename)
    with open(dst, 'wb') as f:
        f.write(data)
    print(f"  Saved: {dst}")

# Show all files
all_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
])
print(f"\nTotal audio files: {len(all_files)}")

### B2. Create `.scp` file and run inference

In [ ]:
import os

# Create .scp file (one audio path per line)
AUDIO_DIR = '/content/audio'
SCP_PATH = '/content/audio_list.scp'
OUTPUT_DIR = '/content/SongFormer/output'
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')

audio_files = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
])

if not audio_files:
    print("ERROR: No audio files found!")
    print("Go back to step 4 and specify a Drive folder, or upload files in B1.")
else:
    with open(SCP_PATH, 'w') as f:
        for path in audio_files:
            f.write(path + '\n')

    print(f"Created {SCP_PATH} with {len(audio_files)} files")
    print("Files:")
    for p in audio_files:
        print(f"  {os.path.basename(p)}")

In [ ]:
%cd /content/SongFormer/src/SongFormer

!python infer/infer.py \
    --input_path /content/audio_list.scp \
    --output_path /content/SongFormer/output \
    --model SongFormer \
    --checkpoint SongFormer.safetensors \
    --config_path SongFormer.yaml \
    --gpu_num 1

print("\n=== Results ===")
import json
for f in sorted(os.listdir('/content/SongFormer/output')):
    if f.endswith('.json'):
        print(f"\n{f}:")
        with open(f'/content/SongFormer/output/{f}') as fh:
            data = json.load(fh)
            for seg in data:
                print(f"  {seg['start']:.1f}s - {seg['end']:.1f}s: {seg['label']}")

### B3. Download results

In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = '/content/SongFormer/output'
ZIP_PATH = '/content/songformer_results.zip'

if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', OUTPUT_DIR)
    print(f"Zipped results to: {ZIP_PATH}")
    files.download(ZIP_PATH)
else:
    print("No results to download. Run inference first (B2).")